In [ ]:
!pip install transformers datasets sentencepiece evaluate rouge_score bert_score accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d87b643706e57b2429a81898a0da928fccf386b4a5e7f870e3f3a83f9fe1d1e7
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import os
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
from bert_score import score as bert_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = "/content/drive/MyDrive/DS310/Dataset/merged_data/"

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
merged_df = pd.read_csv(folder_path + "merged.csv")

In [ ]:
merged_df = merged_df.dropna(subset=["title"]).reset_index(drop=True)

In [ ]:
# Chia train + temp (val + test)
train_df, temp_df = train_test_split(
    merged_df,
    test_size=0.2,
    stratify=merged_df["category"],
    random_state=42
)

# Chia val + test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["category"],
    random_state=42
)

test_df = test_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 9091
Validation: 1136
Test: 1137


## 1. USING THE MODEL WITHOUT FINE TUNING

#### LOADING THE BART MODEL

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    BartTokenizerFast,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import torch

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
article = test_df.loc[1, "full_story"]

# Số từ của bài gốc
orig_len = len(article.split())

# Title nên bằng 1%–2% số từ
max_length = int(orig_len * 0.03)   # 2%
min_length = int(orig_len * 0.02)   # 1%

# Giới hạn an toàn cho summarizer
max_length = max(8, min(max_length, 40))
min_length = max(5, min(min_length, max_length - 1))

print("Orig words:", orig_len)
print("Title max_length:", max_length)
print("Title min_length:", min_length)

inputs = tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        no_repeat_ngram_size=3
    )

title = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Generated title:", title)

Orig words: 555
Title max_length: 16
Title min_length: 11
Generated title: Infant mortality in the United States decreased by 24.2%


In [ ]:
# Clean text
def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\x00", "").replace("\ufffd", "")
    return t.strip()


# Auto-length calculation for TITLE (1–2%)
def auto_length(text):
    orig_len = len(text.split())

    # 1% - 2% of original words
    max_length = int(orig_len * 0.02)   # upper bound (2%)
    min_length = int(orig_len * 0.01)   # lower bound (1%)

    # Safe limits for BART title generation
    max_length = max(8, min(max_length, 40))   # 8 → 40 tokens
    min_length = max(5, min(min_length, max_length - 1))

    return max_length, min_length


# Summarizer with batch
def summarize_batch_safe(texts, batch_size=2):
    summaries = []
    num_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(num_batches):
        batch_texts = texts[i*batch_size : (i+1)*batch_size]

        print(f"\n===== BATCH {i+1}/{num_batches} =====")
        print("Input size:", len(batch_texts))

        cleaned = [clean_text(t) for t in batch_texts]

        # Tokenize safely
        try:
            inputs = tokenizer(
                cleaned,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)
        except Exception as e:
            print("Tokenization failed:", e)
            summaries.extend([""] * len(cleaned))
            continue

        print("Max tokenized length:", inputs["input_ids"].shape[1])

        # Auto-length per text (1–2% words)
        batch_max_lengths = [auto_length(t)[0] for t in cleaned]
        batch_min_lengths = [auto_length(t)[1] for t in cleaned]

        final_max_len = max(batch_max_lengths)
        final_min_len = min(batch_min_lengths)

        print("Generate max_length:", final_max_len)
        print("Generate min_length:", final_min_len)

        # Generate
        try:
            outputs = model.generate(
                **inputs,
                max_length=final_max_len,
                min_length=final_min_len,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0
            )
            batch_sum = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        except Exception as e:
            print("❌ Generation failed:", e)
            batch_sum = [""] * len(cleaned)

        for j, s in enumerate(batch_sum):
            print(f"\n--- Sample {j+1} ---")
            print(s[:300], "...")

        summaries.extend(batch_sum)

    return summaries

# Evaluation (ROUGE + BERTScore)
def evaluate_summaries(refs, preds):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    r1, r2, rl = [], [], []
    for ref, pred in zip(refs, preds):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    # BERTScore
    P, R, F = bert_score(preds, refs, lang="en")

    return {
        "ROUGE-1": 100 * sum(r1)/len(r1),
        "ROUGE-2": 100 * sum(r2)/len(r2),
        "ROUGE-L": 100 * sum(rl)/len(rl),
        "BERTScore": 100 * float(F.mean())
    }

In [ ]:
test_texts = test_df["full_story"].astype(str).tolist()

test_df["bart_title"] = summarize_batch_safe(
    test_texts,
    batch_size=2
)

Streaming output truncated to the last 5000 lines.
===== BATCH 153/569 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 21
Generate min_length: 8

--- Sample 1 ---
Sargassum is a free-floating brown seaweed that plays a vital role ...

--- Sample 2 ---
University of California, Berkeley study finds empathy can reduce suspensions. Students of color were less likely ...

===== BATCH 154/569 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 12
Generate min_length: 6

--- Sample 1 ---
Liquid crystalline elastomers can be used ...

--- Sample 2 ---
Astrocytes, once seen as brain ...

===== BATCH 155/569 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 8
Generate min_length: 5

--- Sample 1 ---
Researchers at the University of ...

--- Sample 2 ---
Ancient apes in Germany co ...

===== BATCH 156/569 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 13
Generate min_length: 5

--- Sample 1 ---
Researchers show how specific i

In [ ]:
metrics = evaluate_summaries(
    refs=test_df["title"].astype(str).tolist(),
    preds=test_df["bart_title"].tolist()
)

print("\n=========== FINAL METRICS ===========")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=========== FINAL METRICS ===========
ROUGE-1: 17.2515
ROUGE-2: 4.9535
ROUGE-L: 15.1971
BERTScore: 84.2370


## 2. FINE - TUNING MODEL

In [ ]:
train_df = train_df[['full_story', 'title']]
val_df = val_df[['full_story', 'title']]
test_df = test_df[['full_story', 'title']]

train = Dataset.from_pandas(train_df)
val = Dataset.from_pandas(val_df)
test = Dataset.from_pandas(test_df)

In [ ]:
from transformers import BartTokenizerFast

tokenizer = BartTokenizerFast.from_pretrained("facebook/bart-base")

MAX_INPUT = 512
MAX_TARGET = 128

def preprocess(examples):
    # Encode input (full_story)
    model_inputs = tokenizer(
        examples["full_story"],
        max_length=MAX_INPUT,
        padding="max_length",
        truncation=True,
    )

    # Encode summary (title)
    labels = tokenizer(
        examples["title"],
        max_length=MAX_TARGET,
        padding="max_length",
        truncation=True,
    )

    # Replace padding token id with -100
    labels["input_ids"] = [
        [(lid if lid != tokenizer.pad_token_id else -100) for lid in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [ ]:
train_tokenized = train.map(
    preprocess,
    batched=True,
    remove_columns=train.column_names
)

val_tokenized = val.map(
    preprocess,
    batched=True,
    remove_columns=val.column_names
)

test_tokenized = test.map(
    preprocess,
    batched=True,
    remove_columns=test.column_names
)

Map:   0%|          | 0/9091 [00:00<?, ? examples/s]

Map:   0%|          | 0/1136 [00:00<?, ? examples/s]

Map:   0%|          | 0/1137 [00:00<?, ? examples/s]

In [ ]:
from transformers import BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
model.to("cuda")

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,

    learning_rate=5e-5,
    num_train_epochs=3,

    predict_with_generate=True,
    fp16=True,

    logging_steps=50,
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

/tmp/ipython-input-3122838089.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23521488 (21522798-uit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,3.556100
100,3.258400
150,3.155400
200,3.156500
250,3.068100
300,2.955000
350,2.926900
400,3.102600
450,2.952800
500,2.989100


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=6819, training_loss=2.0429644390955963, metrics={'train_runtime': 1783.3831, 'train_samples_per_second': 15.293, 'train_steps_per_second': 3.824, 'total_flos': 8314671212789760.0, 'train_loss': 2.0429644390955963, 'epoch': 3.0})

In [ ]:
model_path = "./bart_finetuned/checkpoint-5500"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)

In [ ]:
test_texts = test_df["full_story"].astype(str).tolist()

test_df["bart_finetune_title"] = summarize_batch_safe(
    test_texts,
    batch_size=2
)

Streaming output truncated to the last 5000 lines.
===== BATCH 153/569 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 21
Generate min_length: 8

--- Sample 1 ---
Scientists Uncover 4 decades of Sargassum Blooms in the North Atlantic ...

--- Sample 2 ---
Empathy Interventions Can Strengthen the Racial Gap in School Suspensions ...

===== BATCH 154/569 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 12
Generate min_length: 6

--- Sample 1 ---
New 3D Printing Approach for Shape-Changing ...

--- Sample 2 ---
Your Brain’s Hidden Brain Cell May ...

===== BATCH 155/569 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 8
Generate min_length: 5

--- Sample 1 ---
AI Tool Helps Analy ...

--- Sample 2 ---
Ancient Great Apes Collabor ...

===== BATCH 156/569 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 13
Generate min_length: 5

--- Sample 1 ---
Yellow Fever Vaccination: How the Immune System ...

--- Sample 2 ---
The

In [ ]:
metrics = evaluate_summaries(
    refs=test_df["title"].astype(str).tolist(),
    preds=test_df["bart_finetune_title"].tolist()
)

print("\n=========== FINAL METRICS ===========")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=========== FINAL METRICS ===========
ROUGE-1: 33.3566
ROUGE-2: 16.2421
ROUGE-L: 30.2542
BERTScore: 88.1030


Nhận xét:
Kết quả fine tune khá tốt, có thể triển khai cho ứng dụng thực tế

In [ ]:
!zip -r bart_checkpoint_5500.zip bart_finetuned/checkpoint-5500

  adding: bart_finetuned/checkpoint-5500/ (stored 0%)
  adding: bart_finetuned/checkpoint-5500/rng_state.pth (deflated 26%)
  adding: bart_finetuned/checkpoint-5500/training_args.bin (deflated 53%)
  adding: bart_finetuned/checkpoint-5500/optimizer.pt (deflated 8%)
  adding: bart_finetuned/checkpoint-5500/model.safetensors (deflated 8%)
  adding: bart_finetuned/checkpoint-5500/scheduler.pt (deflated 61%)
  adding: bart_finetuned/checkpoint-5500/generation_config.json (deflated 45%)
  adding: bart_finetuned/checkpoint-5500/trainer_state.json (deflated 77%)
  adding: bart_finetuned/checkpoint-5500/vocab.json (deflated 59%)
  adding: bart_finetuned/checkpoint-5500/special_tokens_map.json (deflated 52%)
  adding: bart_finetuned/checkpoint-5500/tokenizer.json (deflated 82%)
  adding: bart_finetuned/checkpoint-5500/tokenizer_config.json (deflated 75%)
  adding: bart_finetuned/checkpoint-5500/config.json (deflated 64%)
  adding: bart_finetuned/checkpoint-5500/merges.txt (deflated 53%)
  addin